# Étape 4-5 : EDA

Analyse des outliers, distributions et relations entre variables.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

train = pd.read_csv('../data/processed/train_imputed.csv', parse_dates=['date_publication'])
print(f"Train : {train.shape}")

## Étape 4 : Analyse des outliers

In [ ]:
def iqr_bounds(series):
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    return Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

low_p, high_p = iqr_bounds(train['prix'])
low_s, high_s = iqr_bounds(train['surface_m2'])

n_out_p = ((train['prix'] < low_p) | (train['prix'] > high_p)).sum()
n_out_s = ((train['surface_m2'] < low_s) | (train['surface_m2'] > high_s)).sum()

print(f"Outliers prix     : {n_out_p} ({n_out_p/len(train)*100:.1f}%) | borne haute: {high_p:,.0f} MRU")
print(f"Outliers surface  : {n_out_s} ({n_out_s/len(train)*100:.1f}%) | borne haute: {high_s:.0f} m²")
print("\nDécision : NE PAS supprimer — variance réelle dans le contexte mauritanien.")
print("On utilisera log(prix) pour réduire l'asymétrie.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Boxplot prix
axes[0].boxplot(train['prix'] / 1e6, vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.7))
axes[0].set_title('Distribution Prix (MRU)', fontsize=12)
axes[0].set_ylabel('Prix (millions MRU)')

# Boxplot surface
axes[1].boxplot(train['surface_m2'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='darkorange', alpha=0.7))
axes[1].set_title('Distribution Surface (m²)', fontsize=12)
axes[1].set_ylabel('Surface (m²)')

# Scatter prix vs surface
colors = plt.cm.tab10(np.linspace(0, 1, train['quartier'].nunique()))
for i, (q, grp) in enumerate(train.groupby('quartier')):
    axes[2].scatter(grp['surface_m2'], grp['prix'] / 1e6,
                    alpha=0.4, s=20, label=q, color=colors[i])
axes[2].set_xlabel('Surface (m²)')
axes[2].set_ylabel('Prix (M MRU)')
axes[2].set_title('Prix vs Surface', fontsize=12)
axes[2].legend(fontsize=7, ncol=2)

plt.suptitle('Étape 4 : Analyse des Outliers', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../outputs/figures/etape4_outliers.png', dpi=150, bbox_inches='tight')
plt.show()

## Étape 5.1 : Analyse univariée

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Histogramme prix
ax = axes[0, 0]
ax.hist(train['prix'] / 1e6, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(train['prix'].median() / 1e6, color='red', lw=2, label=f"Médiane: {train['prix'].median()/1e6:.1f}M")
ax.set_title('Distribution des Prix', fontsize=12)
ax.set_xlabel('Prix (M MRU)')
ax.legend()

# Histogramme log(1+prix)
ax = axes[0, 1]
ax.hist(np.log1p(train['prix']), bins=40, color='teal', edgecolor='white', alpha=0.8)
ax.axvline(np.log1p(train['prix']).median(), color='red', lw=2)
ax.set_title('Distribution log(1+prix)', fontsize=12)
ax.set_xlabel('log(1+prix)')

# Histogramme surface
ax = axes[0, 2]
ax.hist(train['surface_m2'], bins=40, color='darkorange', edgecolor='white', alpha=0.8)
ax.set_title('Distribution Surface (m²)', fontsize=12)
ax.set_xlabel('Surface (m²)')

# Bar nb_chambres
ax = axes[1, 0]
chambres_counts = train['nb_chambres'].value_counts().sort_index()
ax.bar(chambres_counts.index.astype(str), chambres_counts.values, color='mediumseagreen', edgecolor='white')
ax.set_title('Répartition nb_chambres', fontsize=12)
ax.set_xlabel('Nb chambres')

# Bar nb_salons
ax = axes[1, 1]
salons_counts = train['nb_salons'].value_counts().sort_index()
ax.bar(salons_counts.index.astype(str), salons_counts.values, color='mediumpurple', edgecolor='white')
ax.set_title('Répartition nb_salons', fontsize=12)
ax.set_xlabel('Nb salons')

# Bar quartiers
ax = axes[1, 2]
q_counts = train['quartier'].value_counts()
ax.barh(q_counts.index, q_counts.values, color='coral', edgecolor='white')
ax.set_title('Annonces par quartier', fontsize=12)
ax.set_xlabel("Nombre d'annonces")

plt.suptitle('Étape 5.1 : Analyse Univariée', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../outputs/figures/etape5_univariee.png', dpi=150, bbox_inches='tight')
plt.show()

## Étape 5.2 : Analyse bivariée

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# Boxplot prix par quartier (ordonné par médiane décroissante)
ax = axes[0, 0]
order = train.groupby('quartier')['prix'].median().sort_values(ascending=False).index
data_by_q = [train[train['quartier'] == q]['prix'].values / 1e6 for q in order]
bp = ax.boxplot(data_by_q, labels=order, vert=True, patch_artist=True)
p95 = train['prix'].quantile(0.95) / 1e6
ax.set_ylim(0, p95 * 1.1)
ax.set_title('Prix par quartier (ordonné par médiane)', fontsize=11)
ax.set_ylabel('Prix (M MRU)')
ax.tick_params(axis='x', rotation=45)
for patch in bp['boxes']:
    patch.set_facecolor('steelblue')
    patch.set_alpha(0.7)

# Scatter prix vs surface (top 4 quartiers)
ax = axes[0, 1]
top4 = train['quartier'].value_counts().head(4).index
palette = ['steelblue', 'darkorange', 'mediumseagreen', 'mediumpurple']
for q, color in zip(top4, palette):
    grp = train[train['quartier'] == q]
    ax.scatter(grp['surface_m2'], grp['prix'] / 1e6, alpha=0.5, s=25, label=q, color=color)
ax.set_xlabel('Surface (m²)')
ax.set_ylabel('Prix (M MRU)')
ax.set_title('Prix vs Surface (Top 4 quartiers)', fontsize=11)
ax.legend(fontsize=9)

# Boxplot prix vs nb_chambres
ax = axes[1, 0]
train_ch = train[train['nb_chambres'] <= 10]
order_ch = sorted(train_ch['nb_chambres'].unique())
data_ch = [train_ch[train_ch['nb_chambres'] == c]['prix'].values / 1e6 for c in order_ch]
bp2 = ax.boxplot(data_ch, labels=[str(int(c)) for c in order_ch], patch_artist=True)
ax.set_title('Prix par nb_chambres', fontsize=11)
ax.set_xlabel('Nb chambres')
ax.set_ylabel('Prix (M MRU)')
for patch in bp2['boxes']:
    patch.set_facecolor('darkorange')
    patch.set_alpha(0.7)

# Bar prix médian par quartier
ax = axes[1, 1]
median_by_q = train.groupby('quartier')['prix'].median().sort_values(ascending=False) / 1e6
bars = ax.barh(median_by_q.index, median_by_q.values, color='teal', edgecolor='white')
ax.set_xlabel('Prix médian (M MRU)')
ax.set_title('Prix médian par quartier', fontsize=11)
for bar, val in zip(bars, median_by_q.values):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}M', va='center', fontsize=9)
ax.set_xlim(0, median_by_q.max() * 1.2)

plt.suptitle('Étape 5.2 : Analyse Bivariée', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../outputs/figures/etape5_bivariee.png', dpi=150, bbox_inches='tight')
plt.show()

## Étape 5.3 : Analyse multivariée

In [ ]:
# Heatmap de corrélation
corr_cols = ['prix', 'surface_m2', 'nb_chambres', 'nb_salons', 'nb_sdb']
corr_df = train[corr_cols].dropna()
corr_matrix = corr_df.corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, ax=ax,
            annot_kws={'size': 11})
ax.set_title('Matrice de corrélation', fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/figures/etape5_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print("Corrélations avec prix:")
print(corr_matrix['prix'].drop('prix').sort_values(ascending=False))

In [ ]:
# Tableau statistiques par quartier
stats_q = train.groupby('quartier').agg(
    nb_annonces=('prix', 'count'),
    prix_median=('prix', 'median'),
    prix_moyen=('prix', 'mean'),
    surface_mediane=('surface_m2', 'median'),
    chambres_mediane=('nb_chambres', 'median')
).sort_values('prix_median', ascending=False)

stats_q['prix_median'] = stats_q['prix_median'].map('{:,.0f}'.format)
stats_q['prix_moyen']  = stats_q['prix_moyen'].map('{:,.0f}'.format)
print(stats_q)